# 프롬프트 튜닝 노트북

`src/kbap_review/scoring.py`의 프롬프트를 고치고 결과를 바로 확인하는 용도.

**핵심:** 프롬프트를 이 노트북에 복사하지 않는다. `autoreload`가 걸려 있어서
에디터에서 `scoring.py`를 저장하면 다음 셀 실행 때 자동으로 반영된다.
복사해두면 실제 배치가 쓰는 프롬프트와 노트북이 갈라진다.

**실행 전 — 루트에 `.env` 만들기:**

```bash
cp .env.example .env    # 그리고 GOOGLE_API_KEY 채우기
```

VS Code에서 이 파일을 열고 우상단 커널 선택에서 `.venv`를 고르면 된다.
`.env`는 gitignore 되어 있고, 이미 설정된 환경변수를 덮어쓰지 않는다.

**주의:** 실제 Gemini 호출이라 돈이 든다(음식 1건당 3콜).
그리고 실행하면 출력이 이 `.ipynb` 파일에 저장되니, 커밋 전에 **Clear All Outputs** 할 것.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

from dotenv import load_dotenv

load_dotenv()  # 루트 .env 를 위로 훑어 읽는다 (이 노트북은 notebooks/ 에서 실행된다)

assert os.environ.get("GOOGLE_API_KEY"), "루트에 .env 를 만들고 GOOGLE_API_KEY 를 채우세요 — .env.example 참고"
os.environ.setdefault("KBAP_API_TOKEN", "unused-in-notebook")  # load_config 가 요구하지만 여기선 POST 안 함

In [ ]:
from kbap_review.config import load_config
from kbap_review.graph import build_graph
from kbap_review.scoring import make_scorers


class NullClient:
    """dry_run 이라 호출되지 않지만 build_graph 시그니처를 채운다."""

    async def post_review_result(self, food_id, verdict):
        raise AssertionError("dry_run 인데 POST 가 호출됐다")


config = load_config("../config.yaml")
graph = build_graph(make_scorers(config), NullClient(), config.thresholds, dry_run=True)
config.model, config.avoidance_model, config.thresholds

In [ ]:
# 샘플 — 정상 1건 + 일부러 망가뜨린 1건.
# 망가뜨린 쪽이 제대로 걸리는지가 프롬프트 품질의 진짜 시험이다.

GOOD = {
    "id": 1,
    "koreanName": "김치찌개",
    "description": "돼지고기와 잘 익은 김치를 넣고 끓인 얼큰한 국물 요리입니다.",
    "nameTranslations": {
        "zh-Hans": "泡菜汤", "en": "Kimchi Stew", "ja": "キムチチゲ", "zh-Hant": "泡菜鍋",
        "vi": "Canh kim chi", "id": "Sup Kimchi", "th": "ซุปกิมจิ",
        "ru": "Кимчи-чиге", "es": "Estofado de kimchi",
    },
    "descriptionTranslations": {
        "zh-Hans": "加入猪肉和熟成泡菜炖煮的辣汤",
        "en": "A spicy stew made with pork and well-fermented kimchi",
        "ja": "豚肉と熟成キムチを煮込んだ辛いスープ料理",
        "zh-Hant": "加入豬肉與熟成泡菜燉煮的辣湯",
        "vi": "Món canh cay nấu với thịt heo và kim chi lên men",
        "id": "Sup pedas dengan daging babi dan kimchi fermentasi",
        "th": "ซุปรสเผ็ดต้มกับหมูและกิมจิหมัก",
        "ru": "Острый суп со свининой и квашеной кимчи",
        "es": "Estofado picante con cerdo y kimchi fermentado",
    },
    "spiciness": 7,
    "avoidanceSubstances": [
        {"code": "PORK", "inclusion_percent": 95},
        {"code": "GARLIC", "inclusion_percent": 90},
    ],
    "reviewAttempts": 0,
}

BAD = {
    **GOOD,
    "id": 2,
    "description": "메밀면을 차가운 육수에 말아 먹는 여름 음식입니다.",  # 다른 음식 설명
    "descriptionTranslations": {**GOOD["descriptionTranslations"], "en": "A sweet dessert made with rice cake"},  # 오역
    "avoidanceSubstances": [
        {"code": "PORK", "inclusion_percent": 95},
        {"code": "CRUSTACEAN", "inclusion_percent": 90},  # 생뚱맞은 성분
    ],
}

In [ ]:
async def review(food):
    """그래프 1회 실행 후 판정과 필드별 점수를 보기 좋게 출력."""
    state = await graph.ainvoke({"food": food})
    v = state["verdict"]
    th = config.thresholds

    print(f"[{food['koreanName']}] {v.verdict}   미달: {v.failed_fields or '없음'}")
    for name, score, limit in [
        ("설명", state["description_score"], th.description),
        ("기피성분", state["avoidance_score"], th.avoidance),
    ]:
        mark = "OK" if score.score >= limit else "__"
        print(f"  {mark} {name:6} {score.score:3}  {score.reason}")
    for lang, s in state["translation_scores"].items():
        mark = "OK" if s.score >= th.translations else "__"
        print(f"  {mark} {lang:8} {s.score:3}  {s.reason}")
    if v.review_note:
        print("\n-- 사람에게 보일 사유 --")
        print(v.review_note)
    return state


_ = await review(GOOD)

In [ ]:
# 망가뜨린 쪽 — 설명·en 번역·기피성분 셋 다 걸려야 정상
_ = await review(BAD)

In [ ]:
# 프롬프트 하나만 빠르게 돌리기 — 그래프 안 태우고 해당 스코어러만.
# scoring.py 를 고쳐 저장한 뒤 이 셀만 다시 실행하면 된다(autoreload).
from kbap_review.scoring import avoidance_prompt, make_scorers

scorers = make_scorers(config)

print(avoidance_prompt(BAD))
print("\n--- 결과 ---")
await scorers.avoidance(BAD)

In [ ]:
# 임계값 보정용 — 여러 건 돌려 점수 분포를 본다.
# 정상 건이 70 언저리에 몰리면 임계값이 높은 것이고,
# 망가뜨린 건이 70을 넘으면 프롬프트가 무른 것이다.
import asyncio

SAMPLES = [GOOD, BAD]  # 실제 DB 에서 뽑은 건들로 늘려가며 확인

states = await asyncio.gather(*(graph.ainvoke({"food": f}) for f in SAMPLES))
for food, st in zip(SAMPLES, states):
    tr = st["translation_scores"]
    print(
        f"{food['koreanName']:8} {st['verdict'].verdict:6}"
        f"  설명 {st['description_score'].score:3}"
        f"  기피성분 {st['avoidance_score'].score:3}"
        f"  번역 min {min(s.score for s in tr.values()):3}"
        f" / avg {sum(s.score for s in tr.values()) // len(tr):3}"
    )